In [1]:
import pyspark
from pyspark.sql import SparkSession
import pyarrow.parquet as pq

In [2]:
spark = SparkSession.builder \
    .master("local[*]")\
    .appName('test')\
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/03/06 21:59:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [11]:
spark.version

'4.1.1'

In [3]:
#download the data
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-06 22:00:07--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 

99.84.245.141, 99.84.245.193, 99.84.245.9, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|99.84.245.141|:443... 

connected.


HTTP request sent, awaiting response... 

200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

          yellow_tr   0%[                    ]       0  --.-KB/s               

...

_tripdata_2025-11.p  68%[============>       ]  46.73M  13.9MB/s    eta 2s     

tripdata_2025-11.pa  81%[===============>    ]  55.06M  16.1MB/s    eta 2s     

ripdata_2025-11.par  92%[=================>  ]  62.67M  18.0MB/s    eta 2s     

ipdata_2025-11.parq  99%[==================> ]  67.30M  19.1MB/s    eta 2s     

yellow_tripdata_202 100%[===================>]  67.84M  19.1MB/s    in 4.2s    

2026-03-06 22:00:12 (16.3 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [4]:
#read the data from downloaded file
df = spark.read \
    .option("header", "true") \
    .parquet('yellow_tripdata_2025-11.parquet')

In [5]:
#repartition
df \
    .repartition(4)\
    .write.parquet('data/yellow/2025/11')

In [12]:
#parquet average size
!ls -lh data/yellow/2025/11

total 98M
-rw-r--r-- 1 admin admin 25M Mar  6 22:27 part-00000-78d4d1e1-ad00-43a2-895a-92a676d6e915-c000.snappy.parquet
-rw-r--r-- 1 admin admin 25M Mar  6 22:27 part-00001-78d4d1e1-ad00-43a2-895a-92a676d6e915-c000.snappy.parquet
-rw-r--r-- 1 admin admin 25M Mar  6 22:27 part-00002-78d4d1e1-ad00-43a2-895a-92a676d6e915-c000.snappy.parquet
-rw-r--r-- 1 admin admin 25M Mar  6 22:27 part-00003-78d4d1e1-ad00-43a2-895a-92a676d6e915-c000.snappy.parquet
-rw-r--r-- 1 admin admin   0 Mar  6 22:27 _SUCCESS


In [ ]:
#read all the partitions from folder
df_yellow = spark.read.parquet('data/yellow/2025/11')

### SQL with spark

In [7]:
#Tell spark dataframe is a table.
df_yellow.registerTempTable('yellow_table')

/home/admin/cursos/data-engineering-zoomcamp/06-batch/.venv/lib/python3.11/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [8]:
#sql query test
spark.sql("""
SELECT * FROM yellow_table LIMIT 10;
""").show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-11-07 18:37:45|  2025-11-07 18:41:51|              1|         0.78|         1|                 N|         140|    

In [13]:
#Count records. How many taxi trips were there on the 15th of November. Consider only trips that started on the 15th of November.
spark.sql("""
SELECT COUNT(*)
FROM yellow_table
WHERE DATE(tpep_pickup_datetime) = '2025-11-15'
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [14]:
#What is the length of the longest trip in the dataset in hours?
spark.sql("""
SELECT
    MAX(
        (unix_timestamp(tpep_dropoff_datetime) -
         unix_timestamp(tpep_pickup_datetime)) / 3600
    ) AS max_trip_hours
FROM yellow_table
""").show()

+-----------------+
|   max_trip_hours|
+-----------------+
|90.64666666666666|
+-----------------+



In [15]:
#Download zone lookup data table
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-07 04:08:07--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 

99.84.245.9, 99.84.245.141, 99.84.245.157, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|99.84.245.9|:443... 

connected.


HTTP request sent, awaiting response... 

200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-07 04:08:08 (1.36 GB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [16]:
#zone lookup data table
zones = spark.read.option("header", "true").csv("taxi_zone_lookup.csv")

In [17]:
#registering table
zones.createOrReplaceTempView("zones")

In [18]:
#Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?
spark.sql("""
SELECT z.Zone, COUNT(*) as trips
FROM yellow_table y
JOIN zones z
ON y.PULocationID = z.LocationID
GROUP BY z.Zone
ORDER BY trips ASC
LIMIT 1
""").show()

+-------------+-----+
|         Zone|trips|
+-------------+-----+
|Arden Heights|    1|
+-------------+-----+

